# Proof of Concept of the complete Workflow

The POC should show the complete workflow from the API-call for Text up to the creation of Knowledge Graphs of the extracted (meta)data.
Start with the classes ASB extracted in his CatalysisIE Knowledge Graph and maybe extend it with some additional classes.

Steps:
1) Get a basic description for catalytical reactions from wikipedia introductory section (API).
2) Call LogicMill API to get various paper about this catalytic reaction (API).
3) Call various APIs to get as much abstracts as possible for the papers LogicMill suggested.
4) Hand each abstract to Ollama for the automated (meta)data extraction. (Dump all information in a JSON file. One file for each catalytic reaction)
5) Mapping  
6) Create Knowledge Graph
7) Predefine some queries

# Set list for terms which will be analysed by the workflow
- Contact process
- Fischer-Tropsch process
- Fluid catalytic cracking
- Haber-Bosch process
- Hydroformylation
- Methanation
- Ostwald process
- Selective catalytic reduction
- Steam reforming
- Water–gas shift reaction

In [66]:
# Setup
import re

from coremeta4cat_reaction_schema import (
    CatalyticReactionExtraction,
    CatalysisResearchFieldEnum,
    CatalystFormEnum,
    Quantity,
    get_ollama_format_schema,
)

key = "Water–gas shift reaction"
model = "gemma3:4b"

# Filesystem-safe stem for all intermediate files, e.g. "Steam reforming" -> "Steam_reforming".
# Using this everywhere avoids spaces in filenames and keeps the per-cell hand-off consistent.
slug = re.sub(r"[^\w]+", "_", key).strip("_")

In [67]:
# Call Wikipedia for article as initial "abstract" for LogicMill
import requests

def get_wikipedia_plaintext(title: str, lang: str = "en") -> str:
    url = f"https://{lang}.wikipedia.org/api/rest_v1/page/summary/{title}"
    response = requests.get(url, headers={"User-Agent": "MetaExtract4Cat/1.0"})
    response.raise_for_status()
    return response.json().get("extract", "")

wiki_text = get_wikipedia_plaintext(key)
print(wiki_text)

The water–gas shift reaction (WGSR) describes the reaction of carbon monoxide and water vapor to form carbon dioxide and hydrogen:CO + H2O ⇌ CO2 + H2


In [68]:
# Call LogicMill API to get a list of various paper about the catalytic reactions
import json
import os
from pathlib import Path
from urllib3.util import Retry
from requests import Session
from requests.adapters import HTTPAdapter

# API token is loaded from the environment / a repo-root .env file (gitignored).
# Never hard-code secrets in notebooks. See .env.example.
def _load_logicmill_token():
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        for parent in [Path.cwd(), *Path.cwd().parents]:
            env_file = parent / ".env"
            if env_file.exists():
                for line in env_file.read_text(encoding="utf-8").splitlines():
                    line = line.strip()
                    if line and not line.startswith("#") and "=" in line:
                        k, _, v = line.partition("=")
                        os.environ.setdefault(k.strip(), v.strip().strip("\"'"))
                break
    token = os.environ.get("LOGICMILL_TOKEN")
    if not token:
        raise RuntimeError(
            "LOGICMILL_TOKEN is not set. Copy .env.example to .env (repo root) and "
            "add your token, then re-run. Never commit the token."
        )
    return token

TOKEN = _load_logicmill_token()

# API settings
URL = "https://api.logic-mill.net/api/v1/graphql/"

QUERY = """
query embedDocumentAndSimilaritySearch($data: [EncodeDocumentPart], $indices: [String], $amount: Int, $model: String!) {
  encodeDocumentAndSimilaritySearch(
    data: $data
    indices: $indices
    amount: $amount
    model: $model
  ) {
    id
    score
    index
    document {
      type
      title
      url
      doi
      publicationDate
      PatspecterEmbedding
    }
  }
}
"""

# Build session once at module level
_session = Session()
_retries = Retry(
    total=5,
    backoff_factor=0.1,
    status_forcelist=[500, 501, 502, 503, 504, 524]
)
_session.mount("https://", HTTPAdapter(max_retries=_retries))
_session.headers.update({
    "content-type": "application/json",
    "Authorization": f"Bearer {TOKEN}"
})

def search_logicmill(title: str, abstract: str, amount: int = 10, model: str = "patspecter") -> list:
    variables = {
        "model": model,
        "data": [
            {"key": "title", "value": title},
            {"key": "abstract", "value": abstract}
        ],
        "amount": amount,
        "indices": ["publications"]
    }

    r = _session.post(URL, json={"query": QUERY, "variables": variables})

    if r.status_code != 200:
        raise ConnectionError(f"LogicMill API error {r.status_code}: {r.text}")

    response = r.json()

    if "errors" in response:
        raise ValueError(f"GraphQL error: {response['errors']}")

    return response["data"]["encodeDocumentAndSimilaritySearch"]

title = key
abstract = wiki_text

results = search_logicmill(title, abstract, amount=10)
print(json.dumps(results, indent=2))

with open(f"poc_response/response_{slug}.json", "w") as f:
    json.dump(results, f, indent=4)

[
  {
    "document": {
      "PatspecterEmbedding": [],
      "doi": "10.1080/03602458008067535",
      "publicationDate": "1980-01-01",
      "title": "The Water-Gas Shift Reaction",
      "type": "article",
      "url": "https://openalex.org/works/W2078932319"
    },
    "id": "W2078932319",
    "index": "publications",
    "score": 0.98453337
  },
  {
    "document": {
      "PatspecterEmbedding": [],
      "doi": "",
      "publicationDate": "2007-01-01",
      "title": "Hydrogen",
      "type": "article",
      "url": "https://openalex.org/works/W7099948261"
    },
    "id": "W7099948261",
    "index": "publications",
    "score": 0.9844422
  },
  {
    "document": {
      "PatspecterEmbedding": [],
      "doi": "10.20944/preprints202310.1035.v1",
      "publicationDate": "2023-10-17",
      "title": "Recent Advances in Reverse Water Gas Conversion Reaction",
      "type": "preprint",
      "url": "https://openalex.org/works/W4387697028"
    },
    "id": "W4387697028",
    "index

In [69]:
# Deduplicate the LogicMill results before enrichment
# LogicMill can return the same publication under different OpenAlex IDs
# (e.g. one record with a DOI and one without). Without this step the same
# paper would be enriched, extracted and counted twice in the knowledge graph.
# Deduplicate by OpenAlex id, normalized DOI and normalized title (exact and
# near-duplicate). Keep the first (highest-scoring) copy and back-fill a DOI
# onto it if it was missing.
import json
import re
from difflib import SequenceMatcher

# Two normalized titles closer than this ratio are treated as the same paper.
_TITLE_SIMILARITY_THRESHOLD = 0.95


def _normalize_id(openalex_id: str) -> str:
    """Strip the host so 'https://openalex.org/W123' and 'W123' compare equal."""
    if not openalex_id:
        return ""
    return openalex_id.rstrip("/").rsplit("/", 1)[-1].strip().upper()


def _normalize_doi(doi: str) -> str:
    if not doi:
        return ""
    d = doi.lower().strip()
    for prefix in ("https://doi.org/", "http://dx.doi.org/", "https://dx.doi.org/", "doi:"):
        if d.startswith(prefix):
            d = d[len(prefix):]
    return d.strip()

def _normalize_title(title: str) -> str:
    if not title:
        return ""
    t = title.lower()
    t = re.sub(r"<[^>]+>", " ", t)      # drop HTML/XML tags, e.g. <sub>
    t = re.sub(r"&[a-z]+;", " ", t)     # drop HTML entities, e.g. &lt;
    t = re.sub(r"[^a-z0-9]+", " ", t)   # punctuation -> space
    return re.sub(r"\s+", " ", t).strip()

def deduplicate(items: list) -> tuple:
    seen_id = {}      # normalized OpenAlex id -> position in `unique`
    seen_doi = {}     # normalized DOI         -> position in `unique`
    seen_title = {}   # normalized title       -> position in `unique`
    unique = []
    duplicates = []

    def _fuzzy_title_match(title_n):
        """Return position of an existing near-duplicate title, else None."""
        if not title_n:
            return None
        for kept_title, pos in seen_title.items():
            if SequenceMatcher(None, title_n, kept_title).ratio() >= _TITLE_SIMILARITY_THRESHOLD:
                return pos
        return None

    for item in items:
        doc = item.get("document", {})
        id_n = _normalize_id(item.get("id", ""))
        doi_n = _normalize_doi(doc.get("doi", ""))
        title_n = _normalize_title(doc.get("title", ""))

        match = None
        if id_n and id_n in seen_id:
            match = seen_id[id_n]
        elif doi_n and doi_n in seen_doi:
            match = seen_doi[doi_n]
        elif title_n and title_n in seen_title:
            match = seen_title[title_n]
        else:
            match = _fuzzy_title_match(title_n)

        if match is not None:
            # Same paper already kept. Back-fill a DOI if the kept copy lacked one.
            kept_doc = unique[match]["document"]
            if not _normalize_doi(kept_doc.get("doi", "")) and doi_n:
                kept_doc["doi"] = doc.get("doi", "")
                seen_doi[doi_n] = match
            duplicates.append(item)
            continue

        pos = len(unique)
        unique.append(item)
        if id_n:
            seen_id[id_n] = pos
        if doi_n:
            seen_doi[doi_n] = pos
        if title_n:
            seen_title[title_n] = pos

    return unique, duplicates

# Load raw LogicMill results
input_file = f"poc_response/response_{slug}.json"
with open(input_file, "r") as f:
    items = json.load(f)

unique, duplicates = deduplicate(items)

print(f"\n{'='*60}")
print(f"DEDUPLICATION")
print(f"{'='*60}")
print(f"Loaded:             {len(items)}")
print(f"Unique:             {len(unique)}")
print(f"Duplicates removed: {len(duplicates)}")
if duplicates:
    print(f"\nRemoved as duplicates:")
    for d in duplicates:
        doc = d["document"]
        print(f"  - {doc.get('title', '')[:55]}")
        print(f"      id={d.get('id')}, doi={doc.get('doi') or 'N/A'}")

# Save the deduplicated list for the enrichment step
output_file = f"poc_response/response_{slug}.json"
with open(output_file, "w") as f:
    json.dump(unique, f, indent=4)

print(f"\nSaved {len(unique)} unique documents to: {output_file}")
print(f"{'='*60}\n")


DEDUPLICATION
Loaded:             10
Unique:             10
Duplicates removed: 0

Saved 10 unique documents to: poc_response/response_Water_gas_shift_reaction.json



In [70]:
# Call OpenAlex, Semantic Scholar, CrossRef API's to get as many abstracts as possible
import html
import json
import re
import requests
import os
from time import sleep
from typing import Optional
from urllib3.util import Retry
from requests.adapters import HTTPAdapter
from pathlib import Path
# Contact e-mail used for the "polite pool" of the public APIs (OpenAlex etc.).
# Set in the .env-file (key: E-MAIL). See .env.example.
def _load_contact_email():
    # Prefer python-dotenv if installed; otherwise fall back to parsing a .env
    # from the current dir or any parent (the repo root lives above this folder).
    try:
        from dotenv import load_dotenv
        load_dotenv()
    except ImportError:
        pass
    if not os.environ.get("E-MAIL"):
        for parent in [Path.cwd(), *Path.cwd().parents]:
            env_file = parent / ".env"
            if env_file.exists():
                for line in env_file.read_text(encoding="utf-8").splitlines():
                    line = line.strip()
                    if line and not line.startswith("#") and "=" in line:
                        k, _, v = line.partition("=")
                        os.environ.setdefault(k.strip(), v.strip().strip("\"'"))
                break
    email = os.environ.get("E-MAIL")
    if not email:
        raise RuntimeError(
            "E-MAIL is not set. Copy .env.example to .env (repo root) and "
            "add your contact e-mail, then re-run."
        )
    return email

CONTACT_EMAIL = _load_contact_email()


# Shared session with automatic retry/backoff for the public abstract APIs.
_abstract_session = requests.Session()
_abstract_session.headers.update({"User-Agent": f"MetaExtract4Cat/1.0 (mailto:{CONTACT_EMAIL})"})
_abstract_session.mount("https://", HTTPAdapter(max_retries=Retry(
    total=4,
    backoff_factor=0.5,
    status_forcelist=[429, 500, 502, 503, 504],
    respect_retry_after_header=True,
    allowed_methods=frozenset(["GET"]),
)))

# Load the existing response
with open(f"poc_response/response_{slug}.json", "r") as f:
    data = json.load(f)


def _clean_text(s: Optional[str]) -> Optional[str]:
    """Unescape HTML entities and strip XML/HTML tags (e.g. JATS <jats:p>, <sub>)
    before the text is stored / handed to the LLM."""
    if not s:
        return s
    s = html.unescape(s)
    s = re.sub(r"<[^>]+>", " ", s)
    return re.sub(r"\s+", " ", s).strip()


# An abstract is only useful to the LLM if it is actually running text. Some
# sources return a placeholder, a copyright line or a few stray characters
# instead of a real abstract; those must not be stored or extracted.
_MIN_ABSTRACT_CHARS = 100    # shorter than this is almost never a real abstract
_MIN_ABSTRACT_WORDS = 20     # a real abstract has at least this many words
_MIN_ALPHA_RATIO = 0.6       # fraction of letters among non-space chars (catches gibberish)


def _is_valid_abstract(s: Optional[str]) -> bool:
    """Heuristic gate: accept only real running text as an abstract."""
    if not s:
        return False
    s = s.strip()
    if len(s) < _MIN_ABSTRACT_CHARS:
        return False
    if len(s.split()) < _MIN_ABSTRACT_WORDS:
        return False
    non_space = [c for c in s if not c.isspace()]
    if non_space:
        alpha_ratio = sum(c.isalpha() for c in non_space) / len(non_space)
        if alpha_ratio < _MIN_ALPHA_RATIO:
            return False
    return True


# Trying to get abstract from multiple sources
def get_abstract(openalex_id: str, doi: str, title: str) -> Optional[str]:

    # source 1: OpenAlex via OpenAlex-ID
    abstract = _get_from_openalex(openalex_id)
    if abstract:
        return abstract

    # source 2: semantic scholar with DOI
    if doi and doi.strip():
        abstract = _get_from_semantic_scholar_doi(doi)
        if abstract:
            return abstract

    # source 3: crossref with doi
    if doi and doi.strip():
        abstract = _get_from_crossref(doi)
        if abstract:
            return abstract

    # source 4: semantic scholar with title
    if title and title.strip():
        abstract = _get_from_semantic_scholar_title(title)
        if abstract:
            return abstract

    return None

def _get_from_openalex(openalex_id: str) -> Optional[str]:
    try:
        url = f"https://api.openalex.org/works/{openalex_id}"
        response = _abstract_session.get(url, params={"mailto": CONTACT_EMAIL}, timeout=10)
        if response.status_code == 200:
            work = response.json()

            # try abstract inverted index method
            if "abstract_inverted_index" in work and work["abstract_inverted_index"]:
                try:
                    inverted = work["abstract_inverted_index"]
                    if not inverted:
                        return None
                    max_pos = max(max(positions) for positions in inverted.values())
                    abstract_words = [None] * (max_pos + 1)
                    for word, positions in inverted.items():
                        for pos in positions:
                            abstract_words[pos] = word
                    abstract = " ".join(w for w in abstract_words if w)
                    if abstract:
                        return abstract
                except (ValueError, KeyError, AttributeError) as e:
                    pass

            # try direct abstract field
            if "abstract" in work and work["abstract"]:
                return work["abstract"]
    except requests.Timeout:
        print(f"OpenAlex timeout for {openalex_id}")
    except Exception as e:
        print(f"OpenAlex error for {openalex_id}: {type(e).__name__}")

    return None
            

def _get_from_semantic_scholar_doi(doi: str) -> Optional[str]:
    try:
        url = f"https://api.semanticscholar.org/graph/v1/paper/DOI:{doi}?fields=abstract"
        response = _abstract_session.get(url, timeout=10)
        if response.status_code == 200:
            data = response.json()
            if "abstract" in data and data["abstract"]:
                return data["abstract"]
    except requests.Timeout:
        print(f"Semantic Scholar (DOI) timeout for {doi}")
    except Exception as e:
        print(f"Semantic Scholar (DOI) error for {doi}: {type(e).__name__}")
    return None

def _get_from_semantic_scholar_title(title: str) -> Optional[str]:
    try:
        url = "https://api.semanticscholar.org/graph/v1/paper/search"
        params = {"query": title, "fields": "abstract", "limit": 1}
        response = _abstract_session.get(url, params=params, timeout=10)
        if response.status_code == 200:
            data = response.json()
            if data.get("data") and len(data["data"]) > 0:
                paper = data["data"][0]
                if "abstract" in paper and paper["abstract"]:
                    return paper["abstract"]
    except requests.Timeout:
        print(f"Semantic Scholar (title) timeout for {title}")
    except Exception as e:
        print(f"Semantic Scholar (title) error for {title}: {type(e).__name__}")
    return None
    
def _get_from_crossref(doi: str) -> Optional[str]:
    try:
        url = f"https://api.crossref.org/works/{doi}"
        response = _abstract_session.get(url, params={"mailto": CONTACT_EMAIL}, timeout=10)
        if response.status_code == 200:
            data = response.json()
            if data.get("message", {}).get("abstract"):
                return data["message"]["abstract"]
    except requests.Timeout:
        print(f"Crossref timeout for {doi}")
    except Exception as e:
        print(f"Crossref error for {doi}: {type(e).__name__}")
    return None

# Adding abstracts to document
print(type(data))
items = data #['data']['encodeDocumentAndSimilaritySearch'] --> data is already a list
total = len(items)
success_count = 0
failed_count = 0

print(f"\n{'='*60}")
print(f"Processing {total} documents...")
print(f"{'='*60}\n")

for idx, item in enumerate(items, 1):
    openalex_id = item.get('id', '')
    doi = item['document'].get('doi', '').strip()
    title = item['document'].get('title', '')
    
    print(f"[{idx}/{total}] {title[:50]}...")
    print(f"  ID: {openalex_id}, DOI: {doi if doi else 'N/A'}")
    
    abstract = _clean_text(get_abstract(openalex_id, doi, title))
    
    if abstract and _is_valid_abstract(abstract):
        item['document']['abstract'] = abstract
        success_count += 1
        print(f"  ✓ Added abstract ({len(abstract)} chars)")
    else:
        if abstract:
            print(f"  ✗ Rejected abstract (too short / not text: {len(abstract)} chars)")
        else:
            print(f"  ✗ No abstract found")
        item['document']['abstract'] = None
        failed_count += 1
    
    sleep(1.0)  # Be nice to the APIs (Semantic Scholar throttles unauthenticated traffic)

# Save updated JSON
output_file = f'poc_response/response_{slug}.json'
with open(output_file, 'w') as f:
    json.dump(data, f, indent=2)

print(f"\n{'='*60}")
print(f"SUMMARY")
print(f"{'='*60}")
print(f"Total documents: {total}")
print(f"✓ Abstracts added: {success_count} ({success_count/total*100:.1f}%)")
print(f"✗ Abstracts missing: {failed_count} ({failed_count/total*100:.1f}%)")
print(f"\nSaved to: {output_file}")
print(f"{'='*60}\n")

<class 'list'>

Processing 10 documents...

[1/10] The Water-Gas Shift Reaction...
  ID: W2078932319, DOI: 10.1080/03602458008067535
  ✓ Added abstract (201 chars)
[2/10] Hydrogen...
  ID: W7099948261, DOI: N/A
  ✗ Rejected abstract (too short / not text: 105 chars)
[3/10] Recent Advances in Reverse Water Gas Conversion Re...
  ID: W4387697028, DOI: 10.20944/preprints202310.1035.v1
  ✓ Added abstract (519 chars)
[4/10] Kinetic mechanism of the water-gas-shift reaction ...
  ID: W2299040644, DOI: N/A
  ✓ Added abstract (413 chars)
[5/10] Scale-up and Assessment of Water Gas Shifts...
  ID: W294673352, DOI: N/A
  ✓ Added abstract (1005 chars)
[6/10] Water gas shift reaction for hydrogen production a...
  ID: W2984973874, DOI: 10.1016/j.apenergy.2019.114078
  ✓ Added abstract (1655 chars)
[7/10] Water-gas shift conversion using a feed with a low...
  ID: W2041335864, DOI: 10.1016/0920-5861(95)00323-1
Semantic Scholar (title) error for Water-gas shift conversion using a feed with a low ste

In [71]:
# Call Ollama (local LLM) for the metadata extraction.
import json
import ollama
from pydantic import ValidationError

MODEL = model

SYSTEM_PROMPT = """
You are a chemistry metadata extraction assistant for catalysis research.
Extract structured metadata about the catalytic process "{key}" from the text below.

Field guidance:
- reaction_name: use exactly "{key}"
- used_starting_material: substances used to prepare the catalyst/support (precursors), if described
- used_reactant: the input chemicals converted in the reaction (the feed)
- generated_product: the output chemicals of the reaction
- used_catalyst: the catalyst substance(s), e.g. "Ni/Al2O3", "Rh complex" -- name only
- used_solvent: solvent(s) the reaction is carried out in, if any
- has_duration: reaction/residence time as stated, e.g. "2 h", "30 min"
- has_temperature / has_pressure / has_yield / catalyst_quantity: give value + unit when
  a single clean number is stated (e.g. value=300, unit="degC"); if the text gives a
  range or can't be cleanly split, put the exact wording in raw_text instead and leave
  value/unit null
- catalyst_type: the catalysis research field -- one of heterogeneous_catalysis,
  homogeneous_catalysis, biocatalysis, electrocatalysis, photocatalysis,
  hybrid_catalysis, other; leave null if not mentioned
- catalyst_form: the physical form of the catalyst -- one of thin_film, bulk, powdered,
  deposited_sample, supported, other; leave null if not mentioned

Only use information stated in the text. Do not guess or infer values that are not
present. If a field is not mentioned, set it to null.

Respond with a single JSON object only -- no commentary, no markdown code fences.

Text:
{text}
"""

RETRY_SUFFIX = """

Your previous response had this error: {error}
Please correct your answer. Respond with a single corrected JSON object only,
matching the required schema -- no commentary, no markdown code fences.
"""

RESPONSE_SCHEMA = get_ollama_format_schema()

def _strip_json_fence(content: str) -> str:
    """Defensive strip of a ```json fence, in case the model still adds one
    despite schema-constrained decoding."""
    return content.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()

def _call_ollama(prompt: str) -> CatalyticReactionExtraction:
    response = ollama.generate(
        model=MODEL,
        prompt=prompt,
        format=RESPONSE_SCHEMA,  # enforce schema constraint output (CoreMeta4Cat-subset schema)
        options={
            "temperature": 0.0,   # 0.0 = maximal deteministic
            "seed": 42,           # fix seed for reproducability
            "top_p": 1.0,         # no nucleus sampling
            "top_k": 1            # always use the token with the highest probability
        }
    )
    content = _strip_json_fence(response["response"])
    return CatalyticReactionExtraction.model_validate_json(content)

def extract_metadata(text: str, key: str) -> dict:
    """Extract reaction metadata via Ollama, schema-constrained.

    Exactly one retry on schema/validation failure: the validation error is
    appended to the prompt and the model is asked to self-correct. If the
    retry also fails, the ValidationError propagates to the caller, which
    falls back to extracted_metadata=None / error_count -- the batch loop
    never crashes.
    """
    base_prompt = SYSTEM_PROMPT.format(text=_clean_text(text), key=key)
    try:
        result = _call_ollama(base_prompt)
    except ValidationError as first_error:
        retry_prompt = base_prompt + RETRY_SUFFIX.format(error=first_error)
        result = _call_ollama(retry_prompt)
    return result.model_dump(mode="json")

# Load JSON
input_file = f"poc_response/response_{slug}.json"
with open(input_file, "r") as f:
    items = json.load(f)

total = len(items)
success_count = 0
failed_count = 0
error_count = 0

print(f"\n{'='*60}")
print(f"Processing {total} documents...")
print(f"{'='*60}\n")

for idx, item in enumerate(items, 1):
    title = item["document"].get("title", "")
    abstract = item["document"].get("abstract")

    print(f"[{idx}/{total}] {title[:60]}...")

    if not abstract:
        print(f"  — Skipped (no abstract)")
        failed_count += 1
        continue

    try:
        metadata = extract_metadata(abstract, key)
        item["document"]["extracted_metadata"] = metadata
        success_count += 1
        print(f"  ✓ Metadata extracted")
    except ValidationError as e:
        item["document"]["extracted_metadata"] = None
        error_count += 1
        print(f"  ✗ Schema validation error (after retry): {e}")
    except Exception as e:
        item["document"]["extracted_metadata"] = None
        error_count += 1
        print(f"  ✗ Extraction error: {type(e).__name__}: {e}")

# Save updated JSON
output_file = f"poc_response/response_{slug}.json"
with open(output_file, "w") as f:
    json.dump(items, f, indent=2, ensure_ascii=False)

print(f"\n{'='*60}")
print(f"SUMMARY")
print(f"{'='*60}")
print(f"Total:     {total}")
print(f"✓ Success: {success_count}")
print(f"— Skipped: {failed_count} (no abstract)")
print(f"✗ Errors:  {error_count}")
print(f"\nSaved to: {output_file}")
print(f"{'='*60}\n")


Processing 10 documents...

[1/10] The Water-Gas Shift Reaction...
  ✓ Metadata extracted
[2/10] Hydrogen...
  — Skipped (no abstract)
[3/10] Recent Advances in Reverse Water Gas Conversion Reaction...
  ✓ Metadata extracted
[4/10] Kinetic mechanism of the water-gas-shift reaction over K/sub...
  ✓ Metadata extracted
[5/10] Scale-up and Assessment of Water Gas Shifts...
  ✓ Metadata extracted
[6/10] Water gas shift reaction for hydrogen production and carbon ...
  ✓ Metadata extracted
[7/10] Water-gas shift conversion using a feed with a low steam to ...
  — Skipped (no abstract)
[8/10] Conversion of Carbon Monoxide Co With Water And Calculation ...
  ✓ Metadata extracted
[9/10] The Effect of Lanthanum on Iron Based Water-Shift Catalysts...
  ✓ Metadata extracted
[10/10] Water-Gas Shift and Steam Reforming Reactions Over a Rhodium...
  ✓ Metadata extracted

SUMMARY
Total:     10
✓ Success: 8
— Skipped: 2 (no abstract)
✗ Errors:  0

Saved to: poc_response/response_Water_gas_shift_react

In [72]:
# Mapping LLM-Keys
import json

# Map the LLM metadata keys to concepts from various ontologies (key-level mapping).
# The extracted *values* stay literals for now; only the *fields* get a concept IRI.
from typing import Optional

KEY_MAPPING = {
    "reaction_name": "VOC4CAT:0007009",              # unchanged from before
    "used_starting_material": "RO:0004009",
    "used_reactant": "RO:0004009",   # same predicate as used_starting_material -- this mirrors
                                       # CoreMeta4Cat's own generated slot_uri (both resolve to the
                                       # same RO term in the real schema), not a copy-paste bug.
    "generated_product": "RO:0004008",
    "used_catalyst": "RXNO:0000425",
    "used_solvent": "prov:wasAssociatedWith",
    "has_duration": "schema:duration",
    "has_temperature": "SIO:000008",   # generic "has quantitative attribute" -- CoreMeta4Cat itself
    "has_pressure": "SIO:000008",      # reuses this same URI for has_temperature/has_pressure/
    "has_yield": "SIO:000008",         # has_yield; not an error, that's how the ontology models it.
    "catalyst_quantity": "coremeta4cat:catalyst_quantity",
    "catalyst_type": "VOC4CAT:0007014",              # unchanged from before
    "catalyst_form": "TODO:catalyst_form",  # Checked 2026-07-24 directly against the live
                                              # coremeta4cat_reaction_ap.yaml on nfdi4cat/CoreMeta4Cat@main:
                                              # the slot exists (range CatalystFormEnum) but CoreMeta4Cat
                                              # itself defines no slot_uri for it -- there is no real IRI
                                              # to add to PREFIXES. "TODO" is deliberately kept out of
                                              # PREFIXES so curie_to_iri() returns None and the
                                              # unknown_prefix_fields counter/warning below flags it;
                                              # the *value* still passes through into mapped_metadata
                                              # regardless. Re-check upstream if this ever gets a real
                                              # slot_uri.
}

# CURIE prefixes -> namespaces. The local part is concatenated as-is, so the
# AFO entries (whose local part already contains 'result#AFR_...') resolve too.
# NOTE: RXNO/MOP/SBO/AFO namespaces are best-effort; verify against the source
# ontologies if needed. The expansion logic stays the same regardless.
PREFIXES = {
    "RXNO":    "http://purl.obolibrary.org/obo/RXNO_",
    "MOP":     "http://purl.obolibrary.org/obo/MOP_",
    "SBO":     "http://biomodels.net/SBO/SBO_",
    "VOC4CAT": "https://w3id.org/nfdi4cat/voc4cat_",          # from voc4cat Prefix Sheet
    "AFO":     "http://purl.allotrope.org/ontologies/",        # local part holds 'result#AFR_...'
    "mex":     "https://w3id.org/nfdi4cat/metaextract4cat#",   # local prefix for instances (TODO: final IRI)
    "RO":      "http://purl.obolibrary.org/obo/RO_",           # OBO Relation Ontology (standard)
    "SIO":     "http://semanticscience.org/resource/SIO_",     # standard SIO namespace
    "prov":    "http://www.w3.org/ns/prov#",                   # W3C PROV-O (standard)
    "schema":  "https://schema.org/",                          # schema.org (standard)
    "coremeta4cat": "https://w3id.org/nfdi4cat/coremeta4cat/", # verified directly against
                                                                # coremeta4cat_common.yaml's own
                                                                # `prefixes:` block
}


def curie_to_iri(curie: str) -> Optional[str]:
    """Expand a CURIE ('PREFIX:local') to a full IRI, or None on unknown prefix."""
    prefix, sep, local = curie.partition(":")
    ns = PREFIXES.get(prefix)
    if not sep or ns is None:
        return None
    return ns + local


# Apply the key-level mapping to every document that has extracted metadata.
input_file = f"poc_response/response_{slug}.json"
with open(input_file, "r") as f:
    items = json.load(f)

total = len(items)
mapped_docs = 0
skipped_docs = 0
unknown_prefix_fields = 0

print(f"\n{'='*60}")
print(f"MAPPING")
print(f"{'='*60}")
print(f"Applying KEY_MAPPING to {total} documents...\n")

for item in items:
    doc = item["document"]
    metadata = doc.get("extracted_metadata")

    if not metadata:
        skipped_docs += 1
        continue

    mapped = {}
    for field, curie in KEY_MAPPING.items():
        iri = curie_to_iri(curie)
        if iri is None:
            unknown_prefix_fields += 1
        mapped[field] = {
            "curie": curie,
            "iri": iri,
            "value": metadata.get(field),   # literal(s) unchanged, incl. null / lists / nested dicts
        }

    doc["mapped_metadata"] = mapped
    mapped_docs += 1

# Save updated JSON
output_file = f"poc_response/response_{slug}.json"
with open(output_file, "w") as f:
    json.dump(items, f, indent=2, ensure_ascii=False)

print(f"{'='*60}")
print(f"SUMMARY")
print(f"{'='*60}")
print(f"Total documents:   {total}")
print(f"✓ Mapped:          {mapped_docs}")
print(f"— Skipped:         {skipped_docs} (no extracted_metadata)")
# As of the 2026-07-24 check against upstream CoreMeta4Cat, catalyst_form is the only
# KEY_MAPPING field with no real slot_uri, so unknown_prefix_fields should equal
# mapped_docs (exactly 1 unresolved field per mapped document). If this count is
# anything other than mapped_docs, that's a real new gap in PREFIXES to investigate.
if unknown_prefix_fields:
    print(f"⚠ Unknown prefix:  {unknown_prefix_fields} field(s) got iri=None (check PREFIXES)")
print(f"\nSaved to: {output_file}")
print(f"{'='*60}\n")


MAPPING
Applying KEY_MAPPING to 10 documents...

SUMMARY
Total documents:   10
✓ Mapped:          8
— Skipped:         2 (no extracted_metadata)
⚠ Unknown prefix:  8 field(s) got iri=None (check PREFIXES)

Saved to: poc_response/response_Water_gas_shift_reaction.json



In [73]:
# Create Knowledge Graph
# Turn the extracted (and mapped) metadata of ALL reactions into one combined
# OWL knowledge graph with owlready2. One `Publication_X` individual (X = running
# index) per document that has extracted metadata; DOI, title and abstract are
# stored as rdfs:comment on it. Chemicals / catalysts are shared (deduplicated
# by normalized name) so the same species connects the publications that
# report it.
#
# Missing / placeholder values ("null", "not specified", ...) are skipped: they
# are neither turned into individuals nor into data properties.
#
# Requires the earlier cells to have run so that `KEY_MAPPING`, `curie_to_iri`
# and `_clean_text` are in scope.
import glob
import json
import re
from owlready2 import (
    World, Thing, ObjectProperty, DataProperty, FunctionalProperty,
    AnnotationProperty,
)

BASE_IRI = "https://w3id.org/nfdi4cat/metaextract4cat"
OUTPUT_KG = "poc_response/catalysis_kg.owl"

# Fresh world so re-running the cell rebuilds the graph from scratch instead of
# accumulating duplicate individuals.
world = World()
onto = world.get_ontology(BASE_IRI)

with onto:
    # --- Classes ---
    class Publication(Thing): pass
    class CatalyticReaction(Thing): pass       # the named reaction (shared per source file)
    class ReactionInstance(Thing): pass        # the reaction as reported in one paper
    class ChemicalSpecies(Thing): pass         # shared: starting materials / reactants / products / solvents
    class Catalyst(Thing): pass                # kept as its own class (was already distinct before);
                                                # only its population changes from singular to list-looped

    # --- Annotation property to keep the field -> ontology-concept mapping ---
    class mappedConcept(AnnotationProperty): pass

    # --- Object properties ---
    class reportsReaction(ObjectProperty):
        domain = [Publication]; range = [ReactionInstance]
    class instantiates(ObjectProperty, FunctionalProperty):
        domain = [ReactionInstance]; range = [CatalyticReaction]
    class hasStartingMaterial(ObjectProperty):
        domain = [ReactionInstance]; range = [ChemicalSpecies]
    class hasReactant(ObjectProperty):
        domain = [ReactionInstance]; range = [ChemicalSpecies]
    class hasProduct(ObjectProperty):
        domain = [ReactionInstance]; range = [ChemicalSpecies]
    class hasSolvent(ObjectProperty):
        domain = [ReactionInstance]; range = [ChemicalSpecies]
    class hasCatalyst(ObjectProperty):
        domain = [ReactionInstance]; range = [Catalyst]

    # --- Data properties on Publication ---
    class doi(DataProperty, FunctionalProperty):
        domain = [Publication]; range = [str]
    class openalexId(DataProperty, FunctionalProperty):
        domain = [Publication]; range = [str]
    class score(DataProperty, FunctionalProperty):
        domain = [Publication]; range = [float]
    class publicationDate(DataProperty, FunctionalProperty):
        domain = [Publication]; range = [str]

    # --- Plain scalar data properties on ReactionInstance ---
    class catalystType(DataProperty, FunctionalProperty):
        domain = [ReactionInstance]; range = [str]
    class catalystForm(DataProperty, FunctionalProperty):
        domain = [ReactionInstance]; range = [str]
    class duration(DataProperty, FunctionalProperty):
        domain = [ReactionInstance]; range = [str]

    # --- Quantity data properties (value+unit, with a raw_text fallback for
    # anything the LLM couldn't cleanly split, e.g. ranges) ---
    class temperatureValue(DataProperty, FunctionalProperty):
        domain = [ReactionInstance]; range = [float]
    class temperatureUnit(DataProperty, FunctionalProperty):
        domain = [ReactionInstance]; range = [str]
    class temperatureRawText(DataProperty, FunctionalProperty):
        domain = [ReactionInstance]; range = [str]

    class pressureValue(DataProperty, FunctionalProperty):
        domain = [ReactionInstance]; range = [float]
    class pressureUnit(DataProperty, FunctionalProperty):
        domain = [ReactionInstance]; range = [str]
    class pressureRawText(DataProperty, FunctionalProperty):
        domain = [ReactionInstance]; range = [str]

    class yieldValue(DataProperty, FunctionalProperty):
        domain = [ReactionInstance]; range = [float]
    class yieldUnit(DataProperty, FunctionalProperty):
        domain = [ReactionInstance]; range = [str]
    class yieldRawText(DataProperty, FunctionalProperty):
        domain = [ReactionInstance]; range = [str]

    class catalystQuantityValue(DataProperty, FunctionalProperty):
        domain = [ReactionInstance]; range = [float]
    class catalystQuantityUnit(DataProperty, FunctionalProperty):
        domain = [ReactionInstance]; range = [str]
    class catalystQuantityRawText(DataProperty, FunctionalProperty):
        domain = [ReactionInstance]; range = [str]

# Annotate each mex property with the ontology concept it maps to (reuse of the
# KEY_MAPPING / curie_to_iri from the mapping cell). Keeps the mapping documented
# in the graph without overloading the property identity.
_PROP_TO_FIELD = {
    hasStartingMaterial: "used_starting_material",
    hasReactant: "used_reactant",
    hasProduct: "generated_product",
    hasSolvent: "used_solvent",
    hasCatalyst: "used_catalyst",
    catalystType: "catalyst_type",
    catalystForm: "catalyst_form",
    duration: "has_duration",
    # quantities -- all three sub-properties annotate back to the same
    # CoreMeta4Cat slot/CURIE; they're just three concrete representations
    # of one has_temperature / has_pressure / has_yield / catalyst_quantity value.
    temperatureValue: "has_temperature", temperatureUnit: "has_temperature", temperatureRawText: "has_temperature",
    pressureValue: "has_pressure", pressureUnit: "has_pressure", pressureRawText: "has_pressure",
    yieldValue: "has_yield", yieldUnit: "has_yield", yieldRawText: "has_yield",
    catalystQuantityValue: "catalyst_quantity", catalystQuantityUnit: "catalyst_quantity",
    catalystQuantityRawText: "catalyst_quantity",
}
for _prop, _field in _PROP_TO_FIELD.items():
    _curie = KEY_MAPPING.get(_field)
    if _curie:
        _iri = curie_to_iri(_curie)
        _prop.mappedConcept.append(_iri if _iri else _curie)


# --- Helpers ---------------------------------------------------------------
# The LLM sometimes writes a placeholder string instead of null for a missing
# field. These must be skipped (no individual, no data property). Exact match on
# the normalized value only -- a substring match would wrongly drop real values
# such as "natural gas" or "hydrogenation".
_MISSING_VALUES = {
    "", "null", "none", "n/a", "na", "not specified", "not mentioned",
    "not found", "not available", "not reported", "not stated",
    "unknown", "unspecified", "not applicable",
}


def _is_missing(v) -> bool:
    """True if `v` is empty or a known 'not extracted' placeholder."""
    return not isinstance(v, str) or v.strip().lower() in _MISSING_VALUES


def _safe_name(prefix: str, text: str) -> str:
    """owlready2-safe local name: '<prefix>_<slugified text>'."""
    s = re.sub(r"\W+", "_", (text or "").strip().lower()).strip("_")
    if not s:
        s = "unnamed"
    if s[0].isdigit():
        s = "_" + s
    return f"{prefix}_{s}"


def _norm(text: str) -> str:
    """Normalized key for deduplicating shared entities by name."""
    return re.sub(r"\s+", " ", (text or "").strip().lower())


# One shared cache per entity type: normalized name -> individual.
_chem_cache, _cat_cache = {}, {}


def _get_or_create(cls, cache, prefix, name):
    """Return the shared individual for `name`, creating it once (deduplicated).
    Placeholder / missing names yield no individual (returns None)."""
    if _is_missing(name):
        return None
    name = name.strip()
    k = _norm(name)
    ind = cache.get(k)
    if ind is None:
        ind = cls(_safe_name(prefix, name))
        ind.label.append(name)
        cache[k] = ind
    return ind


# --- Build the graph -------------------------------------------------------
pub_counter = 0
skipped = 0
reaction_nodes = {}   # slug -> shared CatalyticReaction individual

print(f"\n{'='*60}\nKNOWLEDGE GRAPH\n{'='*60}")

for path in sorted(glob.glob("poc_response/response_*.json")):
    file_slug = re.sub(r"^response_|\.json$", "", path.replace("\\", "/").rsplit("/", 1)[-1])
    with open(path, "r", encoding="utf-8") as f:
        items = json.load(f)

    # Shared named-reaction node for this source file.
    reaction_node = reaction_nodes.get(file_slug)
    if reaction_node is None:
        reaction_node = CatalyticReaction(_safe_name("reaction", file_slug))
        reaction_node.label.append(file_slug.replace("_", " "))
        reaction_nodes[file_slug] = reaction_node

    for item in items:
        docd = item.get("document", {})
        meta = docd.get("extracted_metadata")
        if not meta:
            skipped += 1
            continue

        pub_counter += 1
        # Publications are named/labelled 'Publication_<n>' (running index); the
        # paper title is kept as a comment, not as the label.
        pub = Publication(f"Publication_{pub_counter}")
        pub.label.append(f"Publication_{pub_counter}")

        if item.get("id"):
            pub.openalexId = item["id"]
        if isinstance(item.get("score"), (int, float)):
            pub.score = float(item["score"])
        if docd.get("publicationDate"):
            pub.publicationDate = docd["publicationDate"]

        # DOI, title and abstract as rdfs:comment -- only when actually present.
        _doi = (docd.get("doi") or "").strip()
        if _doi:
            pub.doi = _doi
            pub.comment.append(f"DOI: {_doi}")
        _title = _clean_text(docd.get("title", ""))
        if _title:
            pub.comment.append(f"Title: {_title}")
        _abstract = _clean_text(docd.get("abstract"))
        if _abstract:
            pub.comment.append(_abstract)

        # The reaction as reported in this paper.
        inst = ReactionInstance(f"reaction_instance_{pub_counter}")
        pub.reportsReaction.append(inst)
        inst.instantiates = reaction_node

        # Substances (shared/deduplicated per role; placeholders skipped in
        # _get_or_create). used_starting_material / used_reactant /
        # generated_product / used_solvent share ChemicalSpecies (same
        # simplification already applied to reactants/products before);
        # used_catalyst keeps its own Catalyst class, now populated from a
        # list instead of a single string.
        for _field, _cls, _cache, _prefix, _prop_name in (
            ("used_starting_material", ChemicalSpecies, _chem_cache, "chem", "hasStartingMaterial"),
            ("used_reactant", ChemicalSpecies, _chem_cache, "chem", "hasReactant"),
            ("generated_product", ChemicalSpecies, _chem_cache, "chem", "hasProduct"),
            ("used_solvent", ChemicalSpecies, _chem_cache, "chem", "hasSolvent"),
            ("used_catalyst", Catalyst, _cat_cache, "cat", "hasCatalyst"),
        ):
            for name in meta.get(_field) or []:
                ind = _get_or_create(_cls, _cache, _prefix, name)
                if ind:
                    getattr(inst, _prop_name).append(ind)

        # Plain scalar conditions -> data properties (skip missing / placeholder values).
        for _prop_name, _field in (
            ("catalystType", "catalyst_type"),
            ("catalystForm", "catalyst_form"),
            ("duration", "has_duration"),
        ):
            val = meta.get(_field)
            if not _is_missing(val):
                setattr(inst, _prop_name, val.strip())

        # Quantity conditions: value+unit when a clean number was extracted,
        # else the raw_text fallback (never both).
        for _field, _props in (
            ("has_temperature", ("temperatureValue", "temperatureUnit", "temperatureRawText")),
            ("has_pressure", ("pressureValue", "pressureUnit", "pressureRawText")),
            ("has_yield", ("yieldValue", "yieldUnit", "yieldRawText")),
            ("catalyst_quantity", ("catalystQuantityValue", "catalystQuantityUnit", "catalystQuantityRawText")),
        ):
            qty = meta.get(_field)
            if not isinstance(qty, dict):
                continue
            value_prop, unit_prop, raw_prop = _props
            value, unit, raw_text = qty.get("value"), qty.get("unit"), qty.get("raw_text")
            if value is not None:
                setattr(inst, value_prop, float(value))
                if not _is_missing(unit):
                    setattr(inst, unit_prop, unit.strip())
            elif not _is_missing(raw_text):
                setattr(inst, raw_prop, raw_text.strip())

# --- Save & report ---------------------------------------------------------
onto.save(file=OUTPUT_KG, format="rdfxml")

print(f"Publications:        {pub_counter}")
print(f"Skipped (no meta):   {skipped}")
print(f"Named reactions:     {len(reaction_nodes)}")
print(f"Unique chemicals:    {len(_chem_cache)}")
print(f"Unique catalysts:    {len(_cat_cache)}")
print(f"Individuals total:   {len(list(onto.individuals()))}")
print(f"\nSaved knowledge graph to: {OUTPUT_KG}")
print(f"{'='*60}\n")



KNOWLEDGE GRAPH
Publications:        66
Skipped (no meta):   24
Named reactions:     10
Unique chemicals:    108
Unique catalysts:    54
Individuals total:   304

Saved knowledge graph to: poc_response/catalysis_kg.owl

